In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

# 加载数据
train = pd.read_csv('/Users/caierchang/Downloads/train.csv')
test = pd.read_csv('/Users/caierchang/Downloads/test.csv')

def feature_engineering(df):
    # 复制原始数据
    df = df.copy()

    # 1. 处理性别
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

    # 2. 从姓名中提取标题
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
    df['Title'] = df['Title'].map(title_mapping)
    df['Title'] = df['Title'].fillna(0)

    # 3. 处理年龄
    df['Age'] = df.groupby(['Title', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
    df['AgeBin'] = pd.cut(df['Age'].astype(int), 5, labels=False)

    # 4. 家庭特征
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['SmallFamily'] = ((df['FamilySize'] > 1) & (df['FamilySize'] < 5)).astype(int)

    # 5. 票价处理
    df['Fare'] = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))
    df['FareBin'] = pd.qcut(df['Fare'], 4, labels=False)

    # 6. 舱位特征
    df['Pclass_Sex'] = df['Pclass'] * df['Sex']
    df['Pclass_Age'] = df['Pclass'] * df['Age']

    # 7. 登船港口（Embarked）
    df['Embarked'] = df['Embarked'].fillna('S')
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

    # 8. 是否有舱号（可能表示社会地位）
    df['HasCabin'] = df['Cabin'].notna().astype(int)

    # 删除不需要的列
    df.drop(['Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

    return df

# 特征工程
train_processed = feature_engineering(train)
test_processed = feature_engineering(test)

# 选择特征
features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title',
            'FamilySize', 'IsAlone', 'Pclass_Sex', 'HasCabin', 'AgeBin', 'FareBin']

X_train = train_processed[features]
y_train = train_processed['Survived']
X_test = test_processed[features]

# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 使用更好的模型
# 1. 随机森林
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# 2. 梯度提升
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# 交叉验证评估
for model, name in [(rf_model, "Random Forest"), (gb_model, "Gradient Boosting")]:
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"{name} CV Score: {scores.mean():.4f} (+/- {scores.std():.4f})")

# 训练最终模型
final_model = GradientBoostingClassifier(random_state=42)
final_model.fit(X_train_scaled, y_train)

# 预测
predictions = final_model.predict(X_test_scaled)

# 提交文件
output = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predictions})
output.to_csv('/Users/caierchang/Desktop/optimized_submission.csv', index=False)

<>:20: SyntaxWarning: invalid escape sequence '\.'
<>:20: SyntaxWarning: invalid escape sequence '\.'
/var/folders/r9/5f83b4v527g6_r_qsfzx0tbm0000gn/T/ipykernel_6330/933864877.py:20: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


Random Forest CV Score: 0.8283 (+/- 0.0184)
Gradient Boosting CV Score: 0.8305 (+/- 0.0261)
